##### Import des librairies, du CSV

In [27]:
import pandas as pd
import requests
import re

from concurrent.futures import ThreadPoolExecutor
from IPython.display import Image, display
from nutriscope import config
from nutriscope.cli import main
from nutriscope.filtering import output_path_for
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

CSV_GZ = Path("..") / "data" / "france" / "food_france.csv.gz"

dataframe = pd.read_csv(
    CSV_GZ,
    sep="\t",
    usecols=['code', 'product_name', 'image_small_url', 'pnns_groups_1', 'pnns_groups_2'],
    nrows=100000,
    dtype={"code": "string"},
    low_memory=False
)

#### Premier nettoyage du dataset

In [28]:
# Suppression des valeurs manquantes
clean_dataframe = dataframe.dropna(subset=['image_small_url', 'pnns_groups_1', 'pnns_groups_2'])
clean_dataframe = clean_dataframe[(clean_dataframe['pnns_groups_1'] != 'unknown') & (clean_dataframe['pnns_groups_2'] != 'unknown')]

# Suppression des codes doublons
display(clean_dataframe['code'].duplicated().sum())


np.int64(0)

Pas de doublons présents

#### Récupération des produits pour les rayons / sous rayons qu'on souhaite garder

In [29]:
clean_dataframe['label'] = clean_dataframe['pnns_groups_1']
masque = clean_dataframe['pnns_groups_2'] == 'Pizza pies and quiches'
clean_dataframe.loc[masque, 'label'] = 'Pizza pies and quiches'

rayons = [
    'Fat and sauces',
    'Fish Meat Eggs',
    'Cereals and potatoes',
    'Beverages',
    'Pizza pies and quiches'
]

clean_dataframe = clean_dataframe[clean_dataframe['label'].isin(rayons)]

display(clean_dataframe.head())

,code,product_name,pnns_groups_1,pnns_groups_2,image_small_url,label
12,0000554004509,Pain de mie sans gluten,Cereals and potatoes,Bread,https://images.openfoodfacts.org/images/produc...,Cereals and potatoes
13,0000557910210,Flocon d'avoine - complète,Cereals and potatoes,Breakfast cereals,https://images.openfoodfacts.org/images/produc...,Cereals and potatoes
17,0000628006569,L'Envoûtante (Pain),Cereals and potatoes,Bread,https://images.openfoodfacts.org/images/produc...,Cereals and potatoes
18,0000653007548,Baguettine aux olives,Cereals and potatoes,Bread,https://images.openfoodfacts.org/images/produc...,Cereals and potatoes
20,0000682009841,Pain de campagne bio nature,Cereals and potatoes,Bread,https://images.openfoodfacts.org/images/produc...,Cereals and potatoes


#### Equilibrage pour avoir le même nombre d'images par classe

In [30]:
n = 500

tailles = clean_dataframe['label'].value_counts()
print(tailles)
assert tailles[rayons].min() >= n, 'une classe a moins de n images'

equilibre = clean_dataframe.groupby('label', group_keys=False).sample(n=n, random_state=42)
display(equilibre['label'].value_counts())

label
Fish Meat Eggs            13333
Cereals and potatoes       6011
Beverages                  5770
Fat and sauces             3649
Pizza pies and quiches      764
Name: count, dtype: int64


label
Beverages                 500
Cereals and potatoes      500
Fat and sauces            500
Fish Meat Eggs            500
Pizza pies and quiches    500
Name: count, dtype: int64

#### Sauvegarde le fichier qui csv qui va servir à la récupération des images

In [31]:
equilibre[['code', 'label', 'image_small_url']].to_csv('../data/manifest.csv', index=False)

#### Récupération des images à partir des données

In [39]:
HEADERS = {"User-Agent": "FormationIA-Exercice/0.1 (lamarque.alexandre7@gmail.com)"}
OUT_DIR = Path('../data/affiches')

def folder_name(label: str) -> str:
    return re.sub(r"\s+", "-", label.strip().lower())

def download_one(row: tuple) -> str:
    code, label, url = row
    folder = OUT_DIR / folder_name(label)
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / f"{code}.jpg"

    if target.exists():
        return "deja"

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
    except requests.RequestException:
        return "echec"
    target.write_bytes(response.content)

    return "ok"

In [ ]:
manifest = pd.read_csv("../data/manifest.csv")
rows = list(manifest[['code', 'label', 'image_small_url']].itertuples(index=False, name=None))

with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(tqdm(pool.map(download_one, rows), total=len(rows)))

print(pd.Series(results).value_counts())

# Vérification : combien par dossier, et quelles images sont illisibles ?
for folder in sorted(OUT_DIR.iterdir()):
    print(folder.name, len(list(folder.glob("*.jpg"))))

corrompues = []
for path in OUT_DIR.rglob("*.jpg"):
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception:
        corrompues.append(path)
print(f"{len(corrompues)} images illisibles")

  3%|▎         | 75/2500 [03:40<1:59:03,  2.95s/it]
